# ④ 発展編：グループ分け・回帰予測・因果推論
**岡山県 高校教員向け データサイエンス・ハンズオン（発展／③の別案）**

③のグループワークの代わりに選べる「発展コース」です。3つのテーマを扱います。
1. **クラスタリング（K-means）**：似たもの同士をグループに分ける
2. **回帰分析と予測**：複数の要因から結果を予測する（重回帰）
3. **因果推論の入り口**：「相関」と「因果」を区別し、交絡を調整する

> 使うライブラリは `scikit-learn`（Colabに最初から入っています）。難しく見えても数行で試せます。


## 0. 準備

In [ ]:
# === データ読み込みの準備（このセルを最初に実行）===
import pandas as pd, io, os
BASE_URL = "https://raw.githubusercontent.com/yasuyuki-nogami/ds-handson/main/data/"

def load(name):
    if os.path.exists("data/" + name):
        return pd.read_csv("data/" + name)
    if BASE_URL:
        try:
            return pd.read_csv(BASE_URL + name)
        except Exception:
            pass
    from google.colab import files
    print(name, " をアップロードしてください")
    up = files.upload(); key = list(up.keys())[0]
    return pd.read_csv(io.BytesIO(up[key]))
print("準備OK")


In [ ]:
!pip -q install japanize-matplotlib
import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
print("準備OK")


---
# 1. クラスタリング（K-means）：47都道府県をグループ分け
「人口・面積・人口密度」の似ている都道府県を、コンピュータに自動でグループ分けさせます。
正解ラベルを与えずにデータの構造を見つけるので **教師なし学習** と呼びます。


In [ ]:
pref = load("prefecture.csv")

# 使う特徴量（3つ）
X = pref[["population_2020", "area_km2", "density_per_km2"]]

# 単位（人・km²）がバラバラだと大きい数字に引っ張られるので、標準化してそろえる
from sklearn.preprocessing import StandardScaler
Xs = StandardScaler().fit_transform(X)
print("標準化後の形：", Xs.shape)


In [ ]:
from sklearn.cluster import KMeans

# 3つのグループに分ける
km = KMeans(n_clusters=3, n_init=10, random_state=0)
pref["cluster"] = km.fit_predict(Xs)

# 各グループに何県入ったか
print(pref["cluster"].value_counts().sort_index())
pref[["prefecture", "population_2020", "density_per_km2", "cluster"]].head(10)


In [ ]:
# グループごとの平均を見て、各グループの「性格」を読み取る
pref.groupby("cluster")[["population_2020","area_km2","density_per_km2"]].mean().round(0)


In [ ]:
# 散布図で色分け（人口 × 人口密度）
plt.figure(figsize=(7,5))
for c in sorted(pref["cluster"].unique()):
    sub = pref[pref["cluster"] == c]
    plt.scatter(sub["population_2020"]/10000, sub["density_per_km2"], label=f"グループ{c}")
plt.xlabel("人口（万人）"); plt.ylabel("人口密度（人/km²）")
plt.title("K-meansによる都道府県のグループ分け")
plt.legend(); plt.show()


In [ ]:
# 各グループにどの都道府県が入ったか一覧
for c in sorted(pref["cluster"].unique()):
    names = pref.loc[pref["cluster"]==c, "prefecture"].tolist()
    print(f"■ グループ{c}（{len(names)}件）:", "、".join(names))


**読み取りの例**：大都市圏（人口も密度も大）／地方の中規模県／面積が広く density が低い県…のように分かれます。
「いくつのグループに分けるか（`n_clusters`）」を 2 や 4 に変えて、結果がどう変わるか試してみましょう。


---
# 2. 回帰分析と予測（重回帰）
複数の要因から「テストの点数」を予測するモデルを作ります。
※ `class_sample.csv` は練習用の**架空データ**です。


In [ ]:
df = load("class_sample.csv")

features = ["study_min", "sleep_hours", "breakfast", "smartphone_hours"]
X = df[features]
y = df["test_score"]

# データを「学習用」と「テスト用」に分ける（予測の実力を正しく測るため）
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print("学習用", X_train.shape, " テスト用", X_test.shape)


In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_train, y_train)

# 各要因の「効き方」（係数）
import pandas as pd
coef = pd.Series(model.coef_, index=features).round(2)
print("係数（1単位増えると点数が何点変わるか）")
print(coef)
print("切片:", round(model.intercept_, 1))


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
pred = model.predict(X_test)
print("R^2（1に近いほど当てはまり良い）:", round(r2_score(y_test, pred), 3))
print("平均絶対誤差（点）:", round(mean_absolute_error(y_test, pred), 2))

# 予測 vs 実際 を散布図で
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred, alpha=0.4)
plt.plot([0,100],[0,100], "r--")
plt.xlabel("実際の点数"); plt.ylabel("予測した点数"); plt.title("予測の精度")
plt.show()


In [ ]:
# 新しい生徒の点数を予測してみる
new_student = pd.DataFrame([{
    "study_min": 90, "sleep_hours": 7.0, "breakfast": 1, "smartphone_hours": 1.5
}])
print("予測点数:", round(model.predict(new_student)[0], 1))
# → study_min や smartphone_hours を変えて、予測がどう動くか試そう


---
# 3. 因果推論の入り口：「相関」と「因果」は違う
**問い**：「塾に通うと成績は上がる？」

観察データをそのまま比べると間違えることがあります。理由は **交絡（こうらく）**。
ここでは、答えが分かっている**シミュレーションデータ**を作って確かめます。

設定（今回の"真実"）：
- 家庭の教育投資 `Z` が、**塾に通うか `X`** と **成績 `Y`** の両方に影響する（＝交絡因子）
- 塾の**本当の効果は +3点**（この値をあとで当てられるか試す）


In [ ]:
rng = np.random.default_rng(0)
n = 2000
Z = rng.normal(0, 1, n)                       # 家庭の教育投資（見えている想定）
p = 1 / (1 + np.exp(-(1.5*Z)))                # Zが高いほど塾に通いやすい
X = rng.binomial(1, p)                         # 塾に通う=1
true_effect = 3.0                              # ← 塾の"本当の"効果（+3点）
Y = 50 + true_effect*X + 8*Z + rng.normal(0, 5, n)   # 成績（Zの影響が大きい）

sim = pd.DataFrame({"教育投資Z": Z, "塾X": X, "成績Y": Y})
sim.head()


In [ ]:
# ① そのまま比較（交絡を無視）：塾に通う人 vs 通わない人の平均点
naive = sim.groupby("塾X")["成績Y"].mean()
print(naive)
print("素朴な差（見かけの効果）:", round(naive[1] - naive[0], 2), "点")
print("→ 本当の効果 +3点 より大きく出てしまう（Zのせいで水増し）")


In [ ]:
# ② 交絡因子Zを"調整"して比較（重回帰でZを入れる）
from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(sim[["塾X", "教育投資Z"]], sim["成績Y"])
print("Zを調整した塾Xの効果:", round(m.coef_[0], 2), "点")
print("→ 本当の効果 +3点 に近づく！ これが交絡の調整")


### まとめ：因果を語るときの注意
- **相関があっても因果とは限らない**（交絡・逆factリスク）。
- 観察データでは、**交絡因子を測って調整**することが第一歩（今回は重回帰でZを調整）。
- 最も強い方法は **ランダム化実験（RCT）**：Xを"くじ引き"で割り当てれば、Zの偏りが消える。
- 現実には測れない交絡もある → 「言えること／言えないこと」を丁寧に区別するのがデータリテラシー。

> 生徒への問いかけ例：「このデータだけで"塾に行けば必ず伸びる"と言い切れる？ 何が足りない？」


---
## 発展チャレンジ（時間があれば）
1. K-means の `n_clusters` を 4 に変え、グループの意味が変わるか考える。
2. 重回帰の `features` から `study_min` を抜くと R^2 はどう変わる？
3. シミュレーションの交絡の強さ（`8*Z` の 8）を大きく/小さくして、素朴な差のズレを観察する。
